# Data Perparation

We will use `HFTBacktest` as both the backtesting framework and the reinforcement learning environment.
Therefore, it is essential to preprocess and convert our market data into the format required by `HFTBacktest`.
Please refer to the official documentation for data format specifications: [HFTBacktest Data Guide](https://hftbacktest.readthedocs.io/en/latest/data.html)


## Load dependencies


In [1]:
import json
import numpy as np
import os
from hftbacktest import event_dtype
from hftbacktest import EVENT_ARRAY, BUY_EVENT, SELL_EVENT, DEPTH_EVENT, TRADE_EVENT, EXCH_EVENT, LOCAL_EVENT

## Set constantes


In [2]:
PATH = 'data/htx_future'
TRADING_PAIRS = [
    'BTC-USDT',
    'ETH-USDT',
    'SOL-USDT',
    'XRP-USDT',
]
OUT_PUT_PATH = 'data/output'
DEPTH = 20

if not os.path.exists(OUT_PUT_PATH):
    os.makedirs(OUT_PUT_PATH)

## Define data preparing methods


In [3]:
def parse_trades(file_path):
    # Create an empty numpy array to store trade events with predefined dtype
    trades = []

    # Open the trade data file (assumed to be JSONL format, i.e. one JSON object per line)
    with open(file_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
                tick = entry.get("tick")

                local_ts = int((entry.get("ts") + 0.5) * 1_000_000)
                data = tick.get("data")

                for trade in data:
                    exch_ts = int(trade.get("ts")) * 1_000_000
                    direction = BUY_EVENT if trade.get(
                        "direction") == "buy" else SELL_EVENT
                    px = float(trade.get("price", 0))
                    qty = float(trade.get("quantity", 0))

                    trades.append((
                        direction | TRADE_EVENT | EXCH_EVENT | LOCAL_EVENT,
                        exch_ts,
                        local_ts,
                        px,
                        qty,
                        0,                # order_id
                        0,                # ival
                        0.0               # faval
                    ))

            except:
                # Skip lines that are not valid JSON or do not match the expected structure
                continue

    trades.sort(key=lambda x: (x[1], x[2]))
    return np.array(trades, dtype=event_dtype)

In [4]:
def parse_orderbook(file_path):
    records = []

    with open(file_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
                tick = entry.get("tick")

                exch_ts = int(tick.get("ts")) * 1_000_000
                local_ts = int((entry.get("ts") + 0.5) * 1_000_000)

                bids = tick.get("bids", [])
                asks = tick.get("asks", [])

                for p, a in bids:
                    records.append((
                        BUY_EVENT | DEPTH_EVENT | EXCH_EVENT | LOCAL_EVENT,
                        exch_ts,
                        local_ts,
                        float(p),
                        float(a),
                        0,                # order_id
                        0,                # ival
                        0.0               # faval
                    ))

                for p, a in asks:
                    records.append((
                        SELL_EVENT | DEPTH_EVENT | EXCH_EVENT | LOCAL_EVENT,
                        exch_ts,
                        local_ts,
                        float(p),
                        float(a),
                        0,                # order_id
                        0,                # ival
                        0.0               # faval
                    ))

            except Exception:
                continue

    records.sort(key=lambda x: (x[1], x[2]))

    return np.array(records, dtype=event_dtype)

### Process Data


In [8]:
orderbook_file_path = 'data/htx_future/ETH-USDT/orderbook_20250607.json'
trade_file_path = 'data/htx_future/ETH-USDT/trade_20250607.json'

orderbook = parse_orderbook(orderbook_file_path)
assert isinstance(orderbook, np.ndarray), "orderbook is not ndarray"
assert orderbook.dtype == event_dtype, "orderbook dtype mismatch"
print("✅ Orderbook Loaded Successfully.")
print("Fields: (ev, exch_ts, local_ts, px, qty, order_id, ival, fval)")
print("Example Orderbook Events:\n", orderbook[:3])
print(f"Total Orderbook Events: {orderbook.shape[0]}\n")

trades = parse_trades(trade_file_path)
assert isinstance(trades, np.ndarray), "trades is not ndarray"
assert trades.dtype == event_dtype, "trades dtype mismatch"
print("✅ Trades Loaded Successfully.")
print("Fields: (ev, exch_ts, local_ts, px, qty, order_id, ival, fval)")
print("Example Trade Events:\n", trades[:3])
print(f"Total Trade Events: {trades.shape[0]}")

✅ Orderbook Loaded Successfully.
Fields: (ev, exch_ts, local_ts, px, qty, order_id, ival, fval)
Example Orderbook Events:
 [(3758096385, 1749351817021000000, 1749351817026500096, 2510.51, 4341., 0, 0, 0.)
 (3758096385, 1749351817021000000, 1749351817026500096, 2510.48,   11., 0, 0, 0.)
 (3758096385, 1749351817021000000, 1749351817026500096, 2510.41,  111., 0, 0, 0.)]
Total Orderbook Events: 595971

✅ Trades Loaded Successfully.
Fields: (ev, exch_ts, local_ts, px, qty, order_id, ival, fval)
Example Trade Events:
 [(3758096386, 1749351814104000000, 1749351814108499968, 2510.48, 3.  , 0, 0, 0.)
 (3758096386, 1749351814104000000, 1749351814108499968, 2510.48, 2.22, 0, 0, 0.)
 (3758096386, 1749351814104000000, 1749351814108499968, 2510.48, 0.4 , 0, 0, 0.)]
Total Trade Events: 1495


In [6]:
input_root = "data/htx_future"
output_root = "data/output"

for symbol in os.listdir(input_root):
    symbol_path = os.path.join(input_root, symbol)
    if not os.path.isdir(symbol_path):
        continue

    output_dir = os.path.join(output_root, symbol)
    os.makedirs(output_dir, exist_ok=True)

    day_files = {}

    for filename in os.listdir(symbol_path):
        full_path = os.path.join(symbol_path, filename)
        if not filename.endswith(".json"):
            continue

        parts = filename.split("_")
        if len(parts) != 2:
            continue
        date = parts[1].replace(".json", "")

        if date not in day_files:
            day_files[date] = {}

        if filename.startswith("orderbook_"):
            day_files[date]["orderbook"] = full_path
        elif filename.startswith("trade_"):
            day_files[date]["trade"] = full_path

    for date, files in day_files.items():
        try:
            all_events = []

            if "orderbook" in files:
                ob = parse_orderbook(files["orderbook"])
                all_events.append(ob)

            if "trade" in files:
                tr = parse_trades(files["trade"])
                all_events.append(tr)

            # 合并为一个 ndarray
            events = np.concatenate(all_events)

            # 按 exch_ts, local_ts 排序（联合排序）
            events.sort(order=["exch_ts", "local_ts"])

            # 保存为 .npz
            save_path = os.path.join(output_dir, f"{date}.npz")
            np.savez(save_path, events=events)
            print(f"Saved {save_path}, total events: {len(events)}")
        except Exception as e:
            print(f"Failed to process {symbol} {date}: {e}")
            exit(1)

Saved data/output/ETH-USDT/20250609.npz, total events: 26609617
Saved data/output/ETH-USDT/20250608.npz, total events: 20140203
Saved data/output/ETH-USDT/20250607.npz, total events: 597466
Saved data/output/XRP-USDT/20250609.npz, total events: 9834447
Saved data/output/XRP-USDT/20250608.npz, total events: 10390794
Saved data/output/XRP-USDT/20250607.npz, total events: 260965
Saved data/output/SOL-USDT/20250609.npz, total events: 16976845
Saved data/output/SOL-USDT/20250608.npz, total events: 15014543
Saved data/output/SOL-USDT/20250607.npz, total events: 469174
Saved data/output/BTC-USDT/20250609.npz, total events: 20203040
Saved data/output/BTC-USDT/20250608.npz, total events: 14658834
Saved data/output/BTC-USDT/20250607.npz, total events: 430253
